# Importação de Pacotes

In [ ]:
#leitura da base de dados
import pandas as pd
from pathlib import Path
import parquet

#extração de elementos textuais
import re

#modelos preditivos escolhidos
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

#validação cruzada
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, HalvingGridSearchCV
import numpy as np

#métricas
import matplotlib
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, classification_report

#pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
def estimadores(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)

    acuracia = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class='ovr',
        average='weighted'
    )

    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test)

    print(f"""
        Acurácia: {acuracia:.3f}
        Recall (weighted): {recall:.3f}
        F1-score (weighted): {f1:.3f}
        ROC AUC (ovr): {roc_auc:.3f}
        """)

    print(classification_report(y_test, y_pred))

## Leitura DataFrame

In [ ]:
df_erro_simples = pd.read_parquet("erro_medico_tidy_final.parquet")
df = df_erro_simples

In [ ]:
df.head(2)

In [ ]:
df.columns

# Aplicação de REGEX à Coluna "resumo_caso"

Ideal de palavras escolhidas:

1) piora
2) agravamento
3) negligência
4) imperícia
5) sequela
5) sofrimento
6) queimadura
7) lesão severa
8) afastamento
9) internação prolongada
10) incapacidade
11) fratura
12) CDC 
13) demora
14) dor intensa/severa
15) cirurgia/cirúrgico


In [ ]:
# piora
padrão_pior = re.compile(r"pior\w+", re.I)

# agravamento
padrão_agrav = re.compile(r"agrav\w+", re.I)

# negligência
padrão_negl = re.compile(r"neglig\w+", re.I)

# imperícia
padrão_imper = re.compile(r"imper[ií]cia", re.I)

# sequela
padrão_sequel = re.compile(r"sequel\w+", re.I)

# sofrimento
padrão_sofrimento = re.compile(r"sofrimento", re.I)

#queimadura
padrão_queimadura = re.compile(r"queimadura", re.I)

# lesão grave/severa
padrão_lesão = re.compile(r"les[aã]o\s(grave|severa)", re.I)

# afastamento
padrão_afast = re.compile(r"afasta[md]\w+", re.I) 

# internação prolongada
padrão_intern = re.compile(r"interna[cç][aã]o\sprolongada", re.I)

# incapacidade
padrão_incap = re.compile(r"incapacidade", re.I)

# fratura
padrão_fratur = re.compile(r"fratur\w+", re.I)

#CDC
padrão_cdc = re.compile(r"cdc|c[oó]digo\sde\sdefesa\sdo\sconsumidor", re.I)

#demora
padrão_demora = re.compile(r"demor\w+", re.I)

#dor intensa/severa
padrão_dor = re.compile(r"dor\s(?:\w+\s)?(intensa|severa|aguda)", re.I)

#cirurgia
padrão_cirurg = re.compile(r"cir[uú]rgi\w+", re.I)

In [ ]:
piora=[]
agrav=[]
negl=[]
imper=[]
sequel=[]
sofrimento=[]
queimadura=[]
lesão=[]
afast=[]
intern=[]
incap=[]
fratura=[]
cdc=[]
demora=[]
dor=[]
cirurg=[]

for i, linha in df.iterrows():
    texto = linha["resumo_caso"]

#1
    resultado_piora = int(bool(re.search(padrão_pior, texto)))
    piora.append(resultado_piora)
#2
    resultado_agrav = int(bool(re.search(padrão_agrav, texto)))
    agrav.append(resultado_agrav)
#3
    resultado_negl = int(bool(re.search(padrão_negl, texto)))
    negl.append(resultado_negl)
#4
    resultado_imper = int(bool(re.search(padrão_imper, texto)))
    imper.append(resultado_imper)
#5
    resultado_sequel = int(bool(re.search(padrão_sequel, texto)))
    sequel.append(resultado_sequel)
#6
    resultado_sofrimento = int(bool(re.search(padrão_sofrimento, texto)))
    sofrimento.append(resultado_sofrimento)
#7
    resultado_queimadura = int(bool(re.search(padrão_queimadura, texto)))
    queimadura.append(resultado_queimadura)
#8
    resultado_lesão = int(bool(re.search(padrão_lesão, texto)))
    lesão.append(resultado_lesão)
#9
    resultado_afast = int(bool(re.search(padrão_afast, texto)))
    afast.append(resultado_afast)
#10
    resultado_intern = int(bool(re.search(padrão_intern, texto)))
    intern.append(resultado_intern)
#11
    resultado_incap = int(bool(re.search(padrão_incap, texto)))
    incap.append(resultado_incap)
#12
    resultado_fratur = int(bool(re.search(padrão_fratur, texto)))
    fratura.append(resultado_fratur)
#13
    resultado_cdc = int(bool(re.search(padrão_cdc, texto)))
    cdc.append(resultado_cdc)
#14
    resultado_demora = int(bool(re.search(padrão_demora, texto)))
    demora.append(resultado_demora)
#15
    resultado_dor = int(bool(re.search(padrão_dor, texto)))
    dor.append(resultado_dor)
#16
    resultado_cirurg = int(bool(re.search(padrão_cirurg, texto)))
    cirurg.append(resultado_cirurg)

### Inclusão de Novas Colunas ao DF

In [ ]:
df["tem_piora"] = piora

df["tem_agrav"] = agrav

df["tem_negl"] = negl 

df["tem_imper"] = imper

df["tem_sequel"] = sequel 

df["tem_sofrimento"] = sofrimento 

df["tem_queimadura"] = queimadura 

df["tem_lesão"] = lesão 

df["tem_afast"] = afast 

df["tem_intern"] = intern 

df["tem_incap"] = incap 

df["tem_fratura"] = fratura 

df["tem_cdc"] = cdc 

df["tem_demora"] = demora 

df["tem_dor"] = dor 

df["tem_cirurg"] = cirurg 

# Aplicação de Pipelines

### GBC

In [ ]:
modelo= GradientBoostingClassifier()

parametros = {
    'modelo__n_estimators': [100],
    'modelo__max_depth': [2, 6],
    'modelo__max_leaf_nodes': [10],
    'modelo__learning_rate': [0.05, 0.2]
    }

In [ ]:
lista_X = ['valor_acao','n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
           'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
           'tem_perito', 'tem_denuncia_lide', 'tem_assistente', 'tem_piora', 'tem_agrav', 
           'tem_negl', 'tem_imper', 'tem_sequel', 'tem_sofrimento', 'tem_queimadura', 
           'tem_lesão', 'tem_afast', 'tem_intern', 'tem_incap', 'tem_fratura', 'tem_cdc', 
           'tem_demora', 'tem_dor', 'tem_cirurg']

X = df[lista_X]

y = df["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)

num_prep = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_prep, lista_X),
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='roc_auc_ovr',
    refit= True,
    cv=5
)

# balanceamento por sample_weight (GradientBoostingClassifier não aceita class_weight)
sample_weight = compute_sample_weight('balanced', y_train)
searchCV_pipeline.fit(X_train, y_train, modelo__sample_weight=sample_weight)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)

### CBC

In [ ]:
modelo= CatBoostClassifier(auto_class_weights='Balanced')

parametros = {
    'modelo__iterations': [300],
    'modelo__depth': [4, 6, 8],
    'modelo__learning_rate': [0.05, 0.1],
    'modelo__verbose': [0]
}

In [ ]:
lista_X = ['valor_acao','n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
           'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
           'tem_perito', 'tem_denuncia_lide', 'tem_assistente', 'tem_piora', 'tem_agrav', 
           'tem_negl', 'tem_imper', 'tem_sequel', 'tem_sofrimento', 'tem_queimadura', 
           'tem_lesão', 'tem_afast', 'tem_intern', 'tem_incap', 'tem_fratura', 'tem_cdc', 
           'tem_demora', 'tem_dor', 'tem_cirurg']

X = df[lista_X]

y = df["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)


pipeline = Pipeline([
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='f1_macro',
    refit= True,
    cv=5
)

searchCV_pipeline.fit(X_train, y_train)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)